# Looming Detection Circuit — Interactive Demo

**Pipeline.** Active hex-grid sampling of the video → PR (divisive
normalisation with MVP feedback) → ON/OFF channels → Borst T4/T5 motion
detectors → LPLC2 looming detector.

# 0. Imports & Input Video

In [ ]:
from pathlib import Path
import sys
import math
from typing import Dict, List
from collections import defaultdict

# Make NeuroCircuitDesk importable from a checkout (no pip install needed)
_THIS_DIR = Path.cwd()
_REPO_ROOT = _THIS_DIR.parent.parent          # docs/demos -> docs -> repo root
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import imageio
from PIL import Image
from tqdm.notebook import tqdm
from ipywidgets import HBox, Output
from IPython.display import display

from neurocircuitdesk.microcircuit import MicroCircuit
from neurocircuitdesk.canvas import Canvas
from neurocircuitdesk.blocks_exe import unified_algorithm
from neurocircuitdesk import state_utils as su
from neurocircuitdesk.io_utils import HexActiveSampleSpring, HexViz

COL_JSON   = _REPO_ROOT / 'neurocircuitdesk' / 'libs' / 'jsons' / 'hexcol_l1m3_new_578.json'
GRAPH_JSON = _REPO_ROOT / 'neurocircuitdesk' / 'libs' / 'jsons' / 'hex_grid_graph.json'
DATA_DIR   = _THIS_DIR / 'data'
OUT_DIR    = _THIS_DIR / 'outputs' / 'looming_notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Load the looming video

The shipped video is a moving-dot stimulus over an outdoor background. We
load it grayscale, normalise to [0, 1] and apply a γ=2.2 decode so the
luminance is roughly linear in photon rate.

In [ ]:
def load_video_frames(video_path, grayscale=True, normalize=True):
    """Load an MP4 into (T, H, W) gamma-decoded numpy array in [0, 1]^2.2."""
    reader = imageio.get_reader(str(video_path), format='ffmpeg')
    frames = []
    for frame in reader:
        if grayscale:
            frame = np.array(Image.fromarray(frame).convert('L'))
        else:
            frame = np.array(Image.fromarray(frame))
        frames.append(frame)
    reader.close()
    video = np.array(frames, dtype=np.float32)
    if normalize:
        video /= 255.0
    return video ** 2.2          # gamma decode -> linear-ish luminance

VIDEO_PATH = DATA_DIR / 'loom_omni-bg.mp4'
og = load_video_frames(VIDEO_PATH, grayscale=True, normalize=True)
print(f'Video: shape={og.shape}, dtype={og.dtype}, '
      f'range=[{og.min():.3f}, {og.max():.3f}]')

### Sample the video through an active hex grid

`HexActiveSampleSpring` places a hex grid of receptive fields over the
video and evolves each RF with spring-damper dynamics driven by luminance.
Returns `(T, N)` per-cell photon-rate-like values; full `(T, N, 4) =
[x, y, R, lum]` state lives on `sampler.state`. We use `radius=20`
(**1261 cells**, matching `N_COLS` below).


In [ ]:
N_COLS = 1261

sampler = HexActiveSampleSpring(
    center_rc=(150, 200),
    video_T=og,
    bio_cols_only=True,
    col_json_path=str(COL_JSON),
    radius=20, window=8, spacing=6, dt=10.0,
)
lum_T = sampler.sample(shift=True, shrink=True)
lum_T = 1e5 * lum_T                                  # scale to photon rates


In [ ]:
# Visualise the active receptive fields overlaid on a frame of the video.
# sampler.values is exactly sampler.state[..., 3] — the luminance channel.
viz = HexViz(
    center_rc=(150, 200),
    video_T=og,
    bio_cols_only=True,
    col_json_path=str(COL_JSON),
    radius=20, window=8, spacing=6,
)
viz.plot_frame(sampler.values, frame_idx=170, title='Active RFs @ t=170')


# I. MicroCircuit Templates
**Construct each stage from primitive blocks (polynomials, division,
temporal filters, rectifiers, multi-port FuncBlocks).**

## 1. Photoreceptor with divisive normalization (PR)

In [ ]:
def pr_dnp_template(mc: MicroCircuit):
    passthrough_pos = (mc.center[0], mc.center[1], mc.center[2] - 0.5)
    mc.add_block('passthrough', *passthrough_pos)

    mc.add_block('T1', *(mc.center[0] - 0.12, mc.center[1], mc.center[2] + 0.75))
    mc.add_block('T2', *(mc.center[0] + 0.12, mc.center[1], mc.center[2] + 0.75))
    mc.add_block('division_block', *mc.center, node_kind='division')

    div_node = mc.get_exec_node('division_block')
    div_node.add_input_port('num_in', port_type='numerator')
    div_node.add_input_port('den_in', port_type='denominator')
    div_node.add_input_port('den_feedback_val',    port_type='denominator',
                            aggregation='weighted_mean')
    div_node.add_input_port('den_feedback_weight', port_type='denominator',
                            aggregation='weighted_mean')

    mc.connect('T1', 'output', 'division_block', 'num_in')
    mc.connect('T2', 'output', 'division_block', 'den_in')

    mc.specify_io(
        inputs=[
            ('input_main',           'T1',             'input'),
            ('input_main',           'T2',             'input'),
            ('input_main',           'passthrough',    'input'),
            ('den_feedback_val',     'division_block', 'den_feedback_val'),
            ('den_feedback_weight',  'division_block', 'den_feedback_weight'),
        ],
        outputs=[
            ('output_main',        'division_block', 'output'),
            ('input_passthrough',  'passthrough',    'output'),
        ],
    )

## 2. ON/OFF channels

In [ ]:

def onoff_template(mc: MicroCircuit):
    mc.add_block('bp_block',  *(mc.center[0],        mc.center[1],
                                mc.center[2] + 0.75), node_kind='temporal_filter')
    mc.add_block('on_block',  *(mc.center[0] - 0.12, mc.center[1],
                                mc.center[2] + 0.25), node_kind='rectifier_pos')
    mc.add_block('off_block', *(mc.center[0] + 0.12, mc.center[1],
                                mc.center[2] + 0.25), node_kind='rectifier_inv')

    mc.connect('bp_block', 'output', 'on_block',  'input')
    mc.connect('bp_block', 'output', 'off_block', 'input')

    mc.specify_io(
        inputs=[('input_main', 'bp_block', 'input')],
        outputs=[
            ('output_main_on',  'on_block',  'output'),
            ('output_main_off', 'off_block', 'output'),
        ],
    )

## 3. MVP — lateral feedback (inter-columnar MIMO)

Each MVP node reads a ring-2 neighbourhood of upstream PR cells and returns,
per neighbour, a `value` and a `weight` that feed back into the PR's
weighted-mean denominator.

In [ ]:
def mvp_microcircuit_template(mc: MicroCircuit, neighborhood: Dict[int, int]):
    input_cols = sorted(neighborhood.keys())
    input_port_names  = [f'input_col_{i}' for i in input_cols]
    output_port_names = ([f'output_val_col_{i}'    for i in input_cols] +
                         [f'output_weight_col_{i}' for i in input_cols])

    mc.add_block(
        'mvp_processor', *mc.center,
        input_names=input_port_names,
        output_names=output_port_names,
    )
    mc.specify_io(
        inputs=[(name, 'mvp_processor', name) for name in input_port_names],
        outputs=[(name, 'mvp_processor', name) for name in output_port_names],
    )

## 4. Motion detector (Borst T4/T5)

A 7-input MISO that reads each column's centre + 6 ring-1 neighbours in
spiral order and produces four directional outputs (a, b, c, d). Stateful —
keeps an N+1 history buffer for the delayed branch.

In [ ]:
def motion_detector_template(mc: MicroCircuit, neighborhood):
    """Seven-input (centre + ring-1) MISO motion-detector host.

    Despite NCD instantiating one detector per column, the cell is
    inter-columnar at the data-flow level — it reads from its centre
    column plus six ring-1 neighbours. We therefore accept ``neighborhood``
    (passed in by ``add_microcircuit_intercolumnar``) for API symmetry,
    but re-derive the spiral column ordering from ``cv.graph_utils``
    inside the body so slot 0 is always the centre.
    """
    del neighborhood  # accepted for iCMC API parity; body uses local_order
    ordered_cols = cv.graph_utils.local_order(
        mc.col_idx, num_rings=1, require_in_graph=False)
    input_port_names  = [f'input_col_{c}' for c in ordered_cols]
    output_port_names = ['output_a', 'output_b', 'output_c', 'output_d']

    mc.add_block(
        'motion_detector_block', *mc.center,
        input_names=input_port_names,
        output_names=output_port_names,
        stateless=False,
    )
    mc.specify_io(
        inputs=[(name, 'motion_detector_block', name) for name in input_port_names],
        outputs=[(name, 'motion_detector_block', name) for name in output_port_names],
    )

## 5. LPLC2-style looming detector

Four dendrite branches (a/b/c/d), each summing motion inputs from a ring-3
neighbourhood centred on a displaced cardinal direction. The axon block
multiplies the four dendrite outputs → looming-selective signal.

In [ ]:
def looming_detector_template(mc: MicroCircuit, neighborhood: Dict[int, int]):
    ordered_cols = cv.graph_utils.local_order(
        mc.col_idx, num_rings=3, require_in_graph=False)

    # Four displaced cardinal sub-neighbourhoods (each a ring-3 around an
    # offset of the centre's ring-3 spiral).
    branch_offsets = {'a': 32, 'b': 23, 'c': 19, 'd': 28}
    branch_cols = {}
    for label, offset in branch_offsets.items():
        if offset < len(ordered_cols):
            branch_cols[label] = cv.graph_utils.local_order(
                ordered_cols[offset], num_rings=3, require_in_graph=False)
        else:
            branch_cols[label] = ordered_cols

    branch_inputs = {label: [f'input_{label}_col_{c}' for c in cols]
                     for label, cols in branch_cols.items()}

    # Four dendrite blocks (multi-input summation)
    for label in 'abcd':
        mc.add_block(f'layer_{label}', *mc.center,
                     input_names=branch_inputs[label],
                     output_names=['output'], stateless=True)

    # Axon block (multiplies the four dendrite outputs)
    mc.add_block('looming_detector_block', *mc.center,
                 input_names=[f'input_{c}' for c in 'abcd'],
                 output_names=['output'], stateless=True)

    for label in 'abcd':
        mc.connect(f'layer_{label}', 'output',
                   'looming_detector_block', f'input_{label}')

    mc.specify_io(
        inputs=([(n, 'layer_a', n) for n in branch_inputs['a']] +
                [(n, 'layer_b', n) for n in branch_inputs['b']] +
                [(n, 'layer_c', n) for n in branch_inputs['c']] +
                [(n, 'layer_d', n) for n in branch_inputs['d']]),
        outputs=[('output_looming', 'looming_detector_block', 'output')],
    )

# II. Algorithms

All algorithms use the **unified backend-agnostic signature** — stateless
`f(inputs, params) -> dict` or stateful `f(inputs, params, state) -> (dict, state)`,
decorated with `@unified_algorithm`. See `docs/unified_algorithm_syntax.md`.


### Polynomials for the photoreceptor's T1/T2 stage

In [ ]:
@unified_algorithm
def T1_poly_demo(inputs, p):
    x = inputs['input']
    return {'output': p['b1'] + p['a1'] * x + p['a2'] * (x * x)}


@unified_algorithm
def T2_poly_demo(inputs, p):
    x = inputs['input']
    return {'output': p['b2'] + p['c1'] * x + p['c2'] * (x * x)}

### Gamma-difference temporal bandpass kernel for ON/OFF

In [ ]:
def bp_filter():
    """Gamma-function bandpass kernel used by the temporal_filter block."""
    def gamma(t, n, tau):
        return ((n * t) ** n * np.exp(-n * t / tau)
                / (math.factorial(n - 1) * tau ** (n + 1))
                * (t[1] - t[0]))
    tH = np.arange(0, 20, 1)
    return gamma(tH, 2, 2) - gamma(tH, 6, 4)

### MVP lateral algorithm (fan-out MIMO with per-neighbour gain)

In [ ]:
def gaussian_kernel_from_distances(neighborhood: Dict[int, int],
                                   sigma: float) -> Dict[int, float]:
    """Per-column Gaussian weight keyed by hex distance. The engine
    auto-packs this dict against the block's output slot axis."""
    return {col: float(np.exp(-0.5 * (dist / sigma) ** 2))
            for col, dist in neighborhood.items()}


@unified_algorithm
def mvp_algorithm_demo(inputs, params):
    """MIMO + fan-out MIMO: per-neighbour delta scaled by neighbourhood mean."""
    F    = inputs['neighbors']         # (n_nbrs,) scalar | (N, max) MLX
    mask = inputs['neighbor_mask']
    g1   = params['g1']                # auto-packed; same shape as F

    n_valid = mask.sum(axis=-1, keepdims=True)
    y       = (F * mask).sum(axis=-1, keepdims=True) / n_valid
    delta   = 1.0165216804198919e-07 * y + 0.001760445128947395 - 0.001

    return {
        'output_val_col_neighbors':    delta * F * 0.33,
        'output_weight_col_neighbors': g1,
    }

### Borst T4/T5 motion algorithm (stateful, ring-buffered)

State is managed through `state_utils.ring_buffer_push/get/len` so the same
body runs on both the scalar engine (deque underneath) and the MLX engine
(rolling `mx.array` underneath).

In [ ]:
@unified_algorithm
def borst_3branch_algorithm(inputs, params, state):
    F       = inputs['neighbors']      # 7 entries: centre + 6 ring-1
    N_delay = params.get('N', 1)
    alpha   = params.get('alpha', 100)
    beta    = params.get('beta', 100)

    buf = su.ring_buffer_push(state, 'history', F, maxlen=N_delay + 1)
    zeros = F[..., 0] * 0.0

    if su.ring_buffer_len(buf, state, 'history') <= N_delay:
        return ({'output_a': zeros, 'output_b': zeros,
                 'output_c': zeros, 'output_d': zeros}, state)

    y_vals  = su.ring_buffer_get(buf, -1)   # newest
    delayed = su.ring_buffer_get(buf,  0)   # oldest in window

    def branch(y_mean, x_mean, z_mean):
        return y_mean * (1.0 + x_mean * alpha) / (1.0 + z_mean * beta)

    y_a = (y_vals[..., 0] + y_vals[..., 1] + y_vals[..., 4]) / 3.0
    y_c = (y_vals[..., 0] + y_vals[..., 5] + y_vals[..., 6]
           + y_vals[..., 2] + y_vals[..., 3]) / 5.0
    y_d = (y_vals[..., 0] + y_vals[..., 2] + y_vals[..., 3]
           + y_vals[..., 5] + y_vals[..., 6]) / 5.0

    d_23  = (delayed[..., 2] + delayed[..., 3]) / 2.0
    d_56  = (delayed[..., 5] + delayed[..., 6]) / 2.0
    d_543 = (delayed[..., 5] + delayed[..., 4] + delayed[..., 3]) / 3.0
    d_126 = (delayed[..., 1] + delayed[..., 2] + delayed[..., 6]) / 3.0

    val_a = branch(y_a, d_23,  d_56)
    val_b = branch(y_a, d_56,  d_23)
    val_c = branch(y_c, d_543, d_126)
    val_d = branch(y_d, d_126, d_543)

    return ({'output_a': val_a, 'output_b': val_b,
             'output_c': val_c, 'output_d': val_d}, state)

### LPLC2 dendrite (sum) and axon (product) algorithms

In [ ]:
@unified_algorithm
def lplc2_dendrite_algorithm_demo(inputs, params):
    """Dendrite branch: sum all incoming directional motion signals."""
    total = 0.0
    for _, val in inputs.items():
        total = total + val
    return {'output': total}


@unified_algorithm
def lplc2_axon_algorithm_demo(inputs, params):
    """Axon: multiply the four dendrite outputs for looming selectivity."""
    return {'output': inputs['input_a'] * inputs['input_b']
                      * inputs['input_c'] * inputs['input_d']}

### Parameter dicts

In [ ]:
DNP_PARAMS   = {'T1': {'b1': 0,   'a1': 0.001, 'a2': 1e-7},
                'T2': {'b2': 1.0, 'c1': 0.001, 'c2': 1e-7}}
BORST_PARAMS = {'N': 2, 'alpha': 100, 'beta': 100}

# III. Compose Circuit

## 1. Setup Canvas

In [ ]:
cv = Canvas(
    w=900, h=700,
    col_json_path=str(COL_JSON),
    interconnect_json_path=str(GRAPH_JSON),
)
for t in ('PR_col', 'MVP', 'ONOFF_col', 'MOTION_ON_col', 'MOTION_OFF_col', 'LOOMING'):
    cv.add_mc_type(t)

## 2. Add MicroCircuits

Five retinotopic layers stacked along the z-axis:

| z     | Layer            | Count                       | Type       |
| ---:  | ---              | ---:                        | ---        |
| +1.3  | PR (DNP)         | 1261                        | columnar   |
| +0.3  | ONOFF            | 1261                        | columnar   |
| −0.3  | MVP              | sparse (ring-2 tiling)      | MIMO       |
| −1.0  | MOTION_ON/OFF    | 1261 × 2                    | columnar   |
| −3.0  | LOOMING          | sparse (ring-6 tiling)      | MIMO       |


In [ ]:
# 2a. Photoreceptors
for col_idx in range(N_COLS):
    cv.add_microcircuit_columnar(col_idx=col_idx, z=1.3,
                                 mc_type='PR_col', template=pr_dnp_template)

# 2b. ON/OFF
for col_idx in range(N_COLS):
    cv.add_microcircuit_columnar(col_idx=col_idx, z=0.3,
                                 mc_type='ONOFF_col', template=onoff_template)

# 2c. MVPs — ring-2 tiling
mvp_centers = cv.graph_utils.calc_mimo_centers(
    limit=N_COLS, step=2, jump=2, num_rings=2, require_in_graph=False)
for col_idx, neighborhood in mvp_centers.items():
    cv.add_microcircuit_intercolumnar(
        mc_type='MVP', center_col_idx=col_idx,
        neighborhood=neighborhood, z=-0.3,
        template=mvp_microcircuit_template)

# 2d. Motion detectors (ON and OFF stacks share centres + ring-1 neighbourhoods).
motion_centers = cv.graph_utils.calc_mimo_centers(
    limit=N_COLS, step=1, jump=1, num_rings=1, require_in_graph=False)
for col_idx, neighborhood in motion_centers.items():
    cv.add_microcircuit_intercolumnar(
        mc_type='MOTION_ON_col', center_col_idx=col_idx,
        neighborhood=neighborhood, z=-1.0,
        template=motion_detector_template)
    cv.add_microcircuit_intercolumnar(
        mc_type='MOTION_OFF_col', center_col_idx=col_idx,
        neighborhood=neighborhood, z=-1.0,
        template=motion_detector_template)

# 2e. Looming detectors — sparser ring-6 tiling
lplc2_centers = cv.graph_utils.calc_mimo_centers(
    limit=N_COLS, step=3, jump=3, num_rings=6, require_in_graph=False)
for col_idx, neighborhood in lplc2_centers.items():
    cv.add_microcircuit_intercolumnar(
        mc_type='LOOMING', center_col_idx=col_idx, z=-3.0,
        neighborhood=neighborhood, template=looming_detector_template)


## 3. Configure MicroCircuits — assign functions and parameters

In [ ]:
# PR cells: T1/T2 polynomial coefficients
for mc in cv.mc_types['PR_col']:
    mc.set_block_func('T1', T1_poly_demo, DNP_PARAMS['T1'])
    mc.set_block_func('T2', T2_poly_demo, DNP_PARAMS['T2'])

# MVPs: lateral algorithm + per-neighbour Gaussian gain (auto-packed by NCD)
for mc in cv.mc_types['MVP']:
    mc.set_block_func('mvp_processor', mvp_algorithm_demo)
    g1 = gaussian_kernel_from_distances(mvp_centers[mc.col_idx], sigma=0.85)
    mc.set_block_params('mvp_processor', {'g1': g1})

# ON/OFF cells: temporal bandpass kernel
for mc in cv.mc_types['ONOFF_col']:
    mc.set_block_params('bp_block', {'filter': bp_filter()})

# Motion detectors: Borst algorithm + alpha/beta parameters
for mc in cv.mc_types['MOTION_ON_col']:
    mc.set_block_func('motion_detector_block', borst_3branch_algorithm)
    mc.set_block_params('motion_detector_block', BORST_PARAMS)
for mc in cv.mc_types['MOTION_OFF_col']:
    mc.set_block_func('motion_detector_block', borst_3branch_algorithm)
    mc.set_block_params('motion_detector_block', BORST_PARAMS)

# LPLC2: dendrites (sum) + axon (product)
for mc in cv.mc_types['LOOMING']:
    for label in 'abcd':
        mc.set_block_func(f'layer_{label}', lplc2_dendrite_algorithm_demo)
    mc.set_block_func('looming_detector_block', lplc2_axon_algorithm_demo)

## 4. Connect MicroCircuits

`cv.connect_utils.{siso, miso, mimo}` schedule inter-MC arrows. Default
draws each arrow into the 3D viz too; pass `skip_viz=True` to suppress
that for large fan-outs.


In [ ]:
# PR -> MVP (forward feedforward into the lateral lateral)
for col_idx in tqdm(mvp_centers.keys(), desc='PR  -> MVP'):
    cv.connect_utils.mimo(
        'PR_col', 'input_passthrough', 'MVP', 'input_col',
        dst_center_col_idx=col_idx,
        cols=list(mvp_centers[col_idx].keys()))

# MVP -> PR feedback (closes the divisive-normalization loop)
for col_idx in tqdm(mvp_centers.keys(), desc='MVP -> PR feedback'):
    cv.connect_utils.mimo(
        'MVP', 'output_val_col', 'PR_col', 'den_feedback_val',
        src_center_col_idx=col_idx,
        cols=list(mvp_centers[col_idx].keys()))
    cv.connect_utils.mimo(
        'MVP', 'output_weight_col', 'PR_col', 'den_feedback_weight',
        src_center_col_idx=col_idx,
        cols=list(mvp_centers[col_idx].keys()))

# PR -> ONOFF (single-column siso)
for col_idx in tqdm(range(N_COLS), desc='PR -> ONOFF'):
    cv.connect_utils.siso('PR_col', 'output_main', 'ONOFF_col', 'input_main',
                          col_idx=col_idx)

# ONOFF -> MOTION (centre + ring-1 miso)
for col_idx in tqdm(range(N_COLS), desc='ONOFF -> MOTION'):
    cv.connect_utils.miso('ONOFF_col', 'output_main_on',
                          'MOTION_ON_col',  'input_col',
                          col_idx=col_idx, num_rings=1)
    cv.connect_utils.miso('ONOFF_col', 'output_main_off',
                          'MOTION_OFF_col', 'input_col',
                          col_idx=col_idx, num_rings=1)

# MOTION -> LOOMING (per-branch ring-3 mimo)
for looming_mc in tqdm(cv.mc_types['LOOMING'], desc='MOTION -> LOOMING'):
    cols_by_layer = defaultdict(list)
    for name in looming_mc.input_ports.keys():
        # name pattern: input_<label>_col_<col_idx>
        _, label, _, col_id = name.split('_', 3)
        cols_by_layer[label].append(int(col_id))
    for label in 'abcd':
        cv.connect_utils.mimo(
            'MOTION_ON_col',  f'output_{label}', 'LOOMING',
            f'input_{label}_col',
            dst_center_col_idx=looming_mc.col_idx,
            cols=cols_by_layer[label])
        cv.connect_utils.mimo(
            'MOTION_OFF_col', f'output_{label}', 'LOOMING',
            f'input_{label}_col',
            dst_center_col_idx=looming_mc.col_idx,
            cols=cols_by_layer[label])

# IV. Visualize the 3D Circuit

Interactive 3D Plotly figure with every block, port, and connection.
`cv.save(path)` writes a standalone HTML file.


In [ ]:
html_path = OUT_DIR / 'looming_circuit'
cv.save(str(html_path))                     # writes <path>.html
html_file = html_path.with_suffix('.html')
size_mb = html_file.stat().st_size / (1024 * 1024)
print(f'Saved interactive 3D circuit -> {html_file}  ({size_mb:.1f} MB)')
print('Open this file in a browser to explore the circuit topology.')

## IV.2  Flat 2D schematic

`cv.gen_flat_diagram()` — publication-style schematic of a 3-column slice.


In [ ]:
fig = cv.gen_flat_diagram(
    cols=3,
    title='Looming circuit — flat schematic (3 cols)',
    save_dir=str(OUT_DIR),
    save_name='looming_flat',
)
print(f'Saved -> {OUT_DIR / "looming_flat.png"}'
      f' and {OUT_DIR / "looming_flat.pdf"}')


In [ ]:
cv.show()

# V. Compile and Execute the Circuit

`Canvas.compile()` flattens every microcircuit into a single executable
graph, detects feedback loops (MVP↔PR), and emits a `Program` that runs on
the scalar NumPy engine. The compiler prints a notice if it inserts a
one-step delay for any algebraic loop.

In [ ]:
prog = cv.compile()


In [ ]:
%time _ = prog.run_program(inputs=lum_T, input_microcircuits=cv.mc_types['PR_col'])

# VI. Probe Circuit For Results

`prog.probe_result(microcircuits, port_name)` returns `(T, N_cols)`.


In [ ]:
am_T = prog.probe_result(cv.mc_types['PR_col'],         'output_main')
on   = prog.probe_result(cv.mc_types['ONOFF_col'],      'output_main_on')
off  = prog.probe_result(cv.mc_types['ONOFF_col'],      'output_main_off')

T4a = prog.probe_result(cv.mc_types['MOTION_ON_col'],  'output_a')
T4b = prog.probe_result(cv.mc_types['MOTION_ON_col'],  'output_b')
T4c = prog.probe_result(cv.mc_types['MOTION_ON_col'],  'output_c')
T4d = prog.probe_result(cv.mc_types['MOTION_ON_col'],  'output_d')

T5a = prog.probe_result(cv.mc_types['MOTION_OFF_col'], 'output_a')
T5b = prog.probe_result(cv.mc_types['MOTION_OFF_col'], 'output_b')
T5c = prog.probe_result(cv.mc_types['MOTION_OFF_col'], 'output_c')
T5d = prog.probe_result(cv.mc_types['MOTION_OFF_col'], 'output_d')

# LOOMING is sparsely tiled — probe_result returns (T, N_cols) with zeros
# at non-centre cols, so plot_frame works without reshaping.
lplc2 = prog.probe_result(cv.mc_types['LOOMING'], 'output_looming')


### Helper: show a row of retinotopic plots side by side

In [ ]:
def show_row(viz, frame_idx, data_list, titles, **plot_kwargs):
    """Render several per-cell scatter overlays at one frame as an HBox."""
    outs = []
    for data, title in zip(data_list, titles):
        out = Output()
        with out:
            viz.plot_frame(data, frame_idx, title=title, **plot_kwargs)
            plt.show()
        outs.append(out)
    display(HBox(outs))


### Input frame and each layer's response at t = 175

In [ ]:
FRAME = min(175, lum_T.shape[0] - 1)

# Input: hex-grid overlay on the original video frame
viz.plot_frame(sampler.values, frame_idx=FRAME, title=f'Input @ t={FRAME}')


In [ ]:
# PR (amacrine-like divisive normalization output)
viz.plot_frame(am_T, frame_idx=FRAME, title=f'PR (DNP) @ t={FRAME}',
                     vmin=0, vmax=1)

In [ ]:
# ON / OFF channels
show_row(viz, FRAME, [on, off], ['ON', 'OFF'], vmin=0, vmax=0.5)

In [ ]:
# T4 directional motion detectors (ON pathway)
show_row(viz, FRAME, [T4a, T4b, T4c, T4d],
         ['T4a', 'T4b', 'T4c', 'T4d'], vmin=0, vmax=0.75)

# T5 directional motion detectors (OFF pathway)
show_row(viz, FRAME, [T5a, T5b, T5c, T5d],
         ['T5a', 'T5b', 'T5c', 'T5d'], vmin=0, vmax=0.75)

In [ ]:
# LPLC2 looming detector — sparser tiling
lplc2_vmax = float(np.nanmax(lplc2)) if np.any(lplc2) else 1.0
viz.plot_frame(lplc2, frame_idx=FRAME,
                     title=f'LPLC2 looming @ t={FRAME}',
                     vmin=0, vmax=lplc2[FRAME].max())

### Save the LPLC2 response as a video (optional, requires FFmpeg)


In [ ]:
# viz.save_frame_video(
#     data=lplc2,
#     fname=str(OUT_DIR / 'lplc2_looming.mp4'),
#     fps=12, title='LPLC2 looming response',
#     vmin=0, vmax=max(1.0, float(np.nanmax(lplc2))),
# )


---

### Next steps

- Swap `prog = cv.compile()` for `vprog = cv.compile_mlx()` to run the same
  circuit on the batched MLX engine (Apple Silicon GPU).
- Re-run with `cv.connect_utils.*(..., skip_viz=True)` if the 3D HTML is
  unwieldy at higher `N_COLS`.
- Replace the `HexActiveSampleSpring` front-end with `FlyEyeSimulatorStatic`
  / `FlyEyeSimulatorActive` for the full optical-pipeline input
  (see `demo_active_sampling.py`).